In [2]:
# This script automatically adjusts to the classifiers used in the main previous script

# ============================================================
# NEXT-ITERATION SCORING SCRIPT
# Scores the next patient(s) using ALL frozen single-classifier models
# found in FROZEN_SINGLE_MODELS/frozen_models_summary.csv
#
# What it does:
# 1) Loads frozen model summary
# 2) Loads next-patient CSV / Excel
# 3) Applies each frozen pipeline to this next patient
# 4) Prints risk scores in Colab
# 5) Saves CSV and Excel with scores and binary predictions
#
# Notes:
# - No classifier names are hard-coded
# - If you reduce RUN_CLASSIFIERS in the training script, this script
#   automatically follows whatever frozen models were actually saved
# - Assumes the frozen joblib contains:
#     pipeline, feature_columns, threshold, classifier_name, model_id
# ============================================================

!pip -q install openpyxl

import os
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin

# ----------------------------
# 1) Mount Google Drive
# ----------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ============================================================
# 2) USER SETTINGS
# ============================================================

# Input folder where frozen models were saved by the training script
FROZEN_MODELS_DIR = "/content/drive/MyDrive/NESTED_ML/results/EXP1/FROZEN_SINGLE_MODELS"

# Frozen summary CSV produced by the training script
FROZEN_SUMMARY_PATH = os.path.join(FROZEN_MODELS_DIR, "frozen_models_summary.csv")

# Input file with next patient(s)
# Can be CSV or Excel
NEW_PATIENTS_PATH = "/content/drive/MyDrive/NESTED_ML/next-patients.xlsx"

# Output folder
OUT_DIR = "/content/drive/MyDrive/NESTED_ML/results/NEXT-PATIENT-1"
os.makedirs(OUT_DIR, exist_ok=True)

# Optional: if your file has these columns, they will be preserved
LABEL_COL_CANDIDATES = ["label", "Label", "LABEL"]

# If True, keep only required model features and metadata cols in saved output
SAVE_COMPACT_OUTPUT = False

# ============================================================
# THRESHOLD OVERRIDE SETTINGS
# If USE_HARDCODED_THRESHOLDS = True, the thresholds below
# override the threshold saved inside each frozen model package.
# Keys should match classifier_name (preferred) or classifier label.
# ============================================================
USE_HARDCODED_THRESHOLDS = True

HARDCODED_THRESHOLDS = {
    "NAIVE_BAYES": 0.987,
    "GAUSSIAN_PROCESS": 0.50,
    "EXTRATREES": 0.56605,
    "LOGREG": 0.70,
    "SGD_LOGLOSS": 0.80,
}

# ============================================================
# 3) LOAD-COMPATIBLE CUSTOM TRANSFORMERS
#    Needed because frozen .joblib models were saved with these
#    class names in the pipeline.
# ============================================================

class UnivariateAUCFilterProper(BaseEstimator, TransformerMixin):
    def __init__(self, high=0.51, low=0.01, invert_low=True, min_keep=50, max_keep=None):
        self.high = float(high)
        self.low = float(low)
        self.invert_low = bool(invert_low)
        self.min_keep = int(min_keep)
        self.max_keep = None if max_keep is None else int(max_keep)

        self.keep_idx_ = None
        self.invert_mask_kept_ = None
        self.auc_ = None
        self.median_ = None

    def fit(self, X, y=None):
        # Included for completeness; frozen inference should normally not call fit.
        X = np.asarray(X, dtype=float)
        n, p = X.shape

        Xc = X.copy()
        Xc[~np.isfinite(Xc)] = np.nan
        med = np.nanmedian(Xc, axis=0)
        med = np.where(np.isfinite(med), med, 0.0)

        self.median_ = med
        self.auc_ = np.full(p, 0.5, dtype=float)
        self.keep_idx_ = np.arange(p, dtype=int)
        self.invert_mask_kept_ = np.zeros(p, dtype=bool)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        Xc = X.copy()
        Xc[~np.isfinite(Xc)] = np.nan

        if getattr(self, "median_", None) is None:
            med = np.nanmedian(Xc, axis=0)
            med = np.where(np.isfinite(med), med, 0.0)
        else:
            med = np.asarray(self.median_, dtype=float)

        Xc = np.where(np.isfinite(Xc), Xc, med)

        keep_idx = getattr(self, "keep_idx_", None)
        if keep_idx is None:
            keep_idx = np.arange(Xc.shape[1], dtype=int)

        Xk = Xc[:, keep_idx]

        invert_mask = getattr(self, "invert_mask_kept_", None)
        if self.invert_low and invert_mask is not None and np.any(invert_mask):
            Xk = Xk.copy()
            Xk[:, np.asarray(invert_mask, dtype=bool)] *= -1.0

        return Xk


class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = float(threshold)
        self.keep_idx_ = None

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.keep_idx_ = np.arange(X.shape[1], dtype=int)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        keep_idx = getattr(self, "keep_idx_", None)
        if keep_idx is None:
            keep_idx = np.arange(X.shape[1], dtype=int)
        return X[:, keep_idx]


class UnionFeatureSelector50(BaseEstimator, TransformerMixin):
    def __init__(self, k=50, min_keep=50, min_votes=2, random_state=42,
                 methods=("N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET", "RFE")):
        self.k = int(k)
        self.min_keep = int(min_keep)
        self.min_votes = int(min_votes)
        self.random_state = int(random_state)
        self.methods = tuple(methods)

        self.keep_idx_ = None
        self.details_ = None

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.keep_idx_ = np.arange(X.shape[1], dtype=int)
        self.details_ = {}
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        keep_idx = getattr(self, "keep_idx_", None)
        if keep_idx is None:
            keep_idx = np.arange(X.shape[1], dtype=int)
        return X[:, keep_idx]


# ============================================================
# 4) HELPERS
# ============================================================

def load_tabular(path):
    path = str(path)
    ext = Path(path).suffix.lower()

    if ext == ".csv":
        return pd.read_csv(path)
    elif ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file extension: {ext}. Use CSV or Excel.")


def find_first_existing_col(columns, candidates):
    cols_lower = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None


def make_true_label_display_series(df, label_col):
    """
    Always returns a display-only true_label column.
    If label column is missing or values are empty/NaN, show 'not available'.
    """
    if label_col is None or label_col not in df.columns:
        return pd.Series(["not available"] * len(df), index=df.index, dtype=object)

    s = df[label_col].copy()
    s = s.astype(object)
    s = s.where(pd.notna(s), "not available")
    s = s.replace(r"^\s*$", "not available", regex=True)

    return s.astype(str)


def predict_score_from_pipeline(estimator, X):
    """
    Returns continuous score / risk score.
    Preference:
      1) predict_proba[:,1]
      2) decision_function
      3) predict
    """
    if hasattr(estimator, "predict_proba"):
        try:
            p = estimator.predict_proba(X)
            p = np.asarray(p)
            if p.ndim == 2 and p.shape[1] >= 2:
                return p[:, 1].astype(float)
            if p.ndim == 1:
                return p.astype(float)
        except Exception:
            pass

    if hasattr(estimator, "decision_function"):
        try:
            s = estimator.decision_function(X)
            return np.asarray(s).ravel().astype(float)
        except Exception:
            pass

    return estimator.predict(X).astype(float)


def load_frozen_model_package(model_path):
    pkg = joblib.load(model_path)
    required = ["pipeline", "feature_columns", "threshold"]
    missing = [k for k in required if k not in pkg]
    if missing:
        raise ValueError(f"Frozen model {model_path} is missing keys: {missing}")
    return pkg


def align_input_to_model(df_raw, feature_columns):
    """
    Creates model input with exactly the expected raw feature columns.
    Missing columns are created as NaN.
    Extra columns are ignored.
    """
    X = df_raw.copy()

    for col in feature_columns:
        if col not in X.columns:
            X[col] = np.nan

    X = X.loc[:, feature_columns].copy()

    # robust numeric coercion
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    return X


def safe_binary_from_threshold(score, thr):
    return (np.asarray(score, dtype=float) >= float(thr)).astype(int)


def resolve_threshold_for_model(pkg, row):
    """
    Returns the active threshold for this model.
    Priority:
      1) HARDCODED_THRESHOLDS[classifier_name or classifier] if enabled
      2) pkg["threshold"] from frozen model
    """
    frozen_thr = float(pkg["threshold"])

    classifier_name = str(pkg.get("classifier_name", row.get("classifier", "")))
    classifier_label = str(row.get("classifier", classifier_name))

    if not USE_HARDCODED_THRESHOLDS:
        return frozen_thr, "frozen"

    for key in (classifier_name, classifier_label):
        if key in HARDCODED_THRESHOLDS:
            return float(HARDCODED_THRESHOLDS[key]), f"hardcoded:{key}"

    raise KeyError(
        f"No hardcoded threshold found for classifier '{classifier_name}' "
        f"(row classifier='{classifier_label}')."
    )


def compute_binary_metrics(y_true, y_pred):
    """
    Metrics that depend on a binary thresholded prediction.
    Returns a dict with accuracy, MCC, sensitivity, specificity, Youden, etc.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true)
    y_true = y_true[valid].astype(int)
    y_pred = y_pred[valid].astype(int)

    if len(y_true) == 0:
        return {
            "n_valid": 0,
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "MCC": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "youden": np.nan,
            "TP": np.nan,
            "TN": np.nan,
            "FP": np.nan,
            "FN": np.nan,
        }

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    n = tp + tn + fp + fn
    accuracy = (tp + tn) / n if n > 0 else np.nan

    sens_den = tp + fn
    spec_den = tn + fp
    sensitivity = tp / sens_den if sens_den > 0 else np.nan
    specificity = tn / spec_den if spec_den > 0 else np.nan

    if np.isfinite(sensitivity) and np.isfinite(specificity):
        balanced_accuracy = (sensitivity + specificity) / 2.0
        youden = sensitivity + specificity - 1.0
    else:
        balanced_accuracy = np.nan
        youden = np.nan

    mcc_den = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    if mcc_den > 0:
        mcc = ((tp * tn) - (fp * fn)) / np.sqrt(mcc_den)
    else:
        mcc = np.nan

    return {
        "n_valid": int(n),
        "accuracy": float(accuracy) if np.isfinite(accuracy) else np.nan,
        "balanced_accuracy": float(balanced_accuracy) if np.isfinite(balanced_accuracy) else np.nan,
        "MCC": float(mcc) if np.isfinite(mcc) else np.nan,
        "sensitivity": float(sensitivity) if np.isfinite(sensitivity) else np.nan,
        "specificity": float(specificity) if np.isfinite(specificity) else np.nan,
        "youden": float(youden) if np.isfinite(youden) else np.nan,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
    }


def maybe_make_mean_ensemble(result_df, model_score_cols):
    """
    Optional simple mean ensemble across frozen-model scores.
    This is NOT z-score ensemble; just a plain arithmetic mean.
    Useful only as an extra descriptive column.
    """
    if len(model_score_cols) == 0:
        return result_df
    result_df["mean_score_across_models"] = result_df[model_score_cols].mean(axis=1)
    return result_df


# ============================================================
# 5) LOAD INPUTS
# ============================================================

print("Loading frozen summary...")
frozen_summary_df = pd.read_csv(FROZEN_SUMMARY_PATH)
display(frozen_summary_df)

if frozen_summary_df.empty:
    raise RuntimeError("Frozen summary is empty. No frozen models found.")

print("Loading new patient file...")
new_df = load_tabular(NEW_PATIENTS_PATH)

print(f"Input shape: {new_df.shape}")
display(new_df.head())

# Always use the first column as patient ID, regardless of header
id_col = new_df.columns[0]

# Label is still detected normally
label_col = find_first_existing_col(new_df.columns, LABEL_COL_CANDIDATES)

print(f"Using ID column: {id_col}")
print(f"Using label column: {label_col if label_col is not None else 'None'}")

# Keep an output copy with original columns
result_df = new_df.copy()

# Always create a display-only true_label column
result_df["true_label"] = make_true_label_display_series(result_df, label_col)


# ============================================================
# 6) SCORE WITH ALL FROZEN MODELS
# ============================================================

score_cols = []
pred_cols = []
used_models_rows = []

for i, row in frozen_summary_df.iterrows():
    clf_name = str(row["classifier"])
    model_path = str(row["save_path"])

    print("\n" + "=" * 90)
    print(f"[{i+1}/{len(frozen_summary_df)}] Loading frozen model: {clf_name}")
    print("=" * 90)

    pkg = load_frozen_model_package(model_path)

    fitted_pipe = pkg["pipeline"]
    feature_columns = list(pkg["feature_columns"])

    model_id = str(pkg.get("model_id", clf_name))
    classifier_name = str(pkg.get("classifier_name", clf_name))

    threshold, threshold_source = resolve_threshold_for_model(pkg, row)

    print(f"Model ID: {model_id}")
    print(f"Classifier: {classifier_name}")
    print(f"Active threshold: {threshold:.6f} ({threshold_source})")
    print(f"Frozen threshold: {float(pkg['threshold']):.6f}")
    print(f"Expected raw features: {len(feature_columns)}")

    X_model = align_input_to_model(new_df, feature_columns)

    missing_cols = [c for c in feature_columns if c not in new_df.columns]
    print(f"Missing input features auto-created as NaN: {len(missing_cols)}")
    if len(missing_cols) > 0:
        print("First missing features:", missing_cols[:20])

    scores = predict_score_from_pipeline(fitted_pipe, X_model)
    preds = safe_binary_from_threshold(scores, threshold)

    score_col = f"score__{classifier_name}"
    pred_col = f"pred__{classifier_name}"

    result_df[score_col] = scores
    result_df[pred_col] = preds

    score_cols.append(score_col)
    pred_cols.append(pred_col)

    used_models_rows.append({
        "classifier": classifier_name,
        "model_id": model_id,
        "model_path": model_path,
        "threshold_used": threshold,
        "threshold_source": threshold_source,
        "frozen_threshold": float(pkg["threshold"]),
        "n_expected_features": len(feature_columns),
        "n_missing_features_in_input": len(missing_cols),
        "val_AUC_mean": row.get("val_AUC_mean", np.nan),
        "val_AUC_std": row.get("val_AUC_std", np.nan),
        "val_MCC": row.get("val_MCC", np.nan),
    })

    out_view = pd.DataFrame({
        "patient_code": result_df[id_col].astype(str),
        "risk_score": pd.to_numeric(result_df[score_col], errors="coerce"),
        "prediction": pd.to_numeric(result_df[pred_col], errors="coerce"),
        "true_label": result_df["true_label"].astype(str),
    })

    print(f"\nVisible scores for {classifier_name}:")
    display(out_view)


# Optional simple mean across models
result_df = maybe_make_mean_ensemble(result_df, score_cols)

used_models_df = pd.DataFrame(used_models_rows)

# If true labels exist, compute per-model correctness columns and threshold-based metrics
metrics_rows = []

if label_col is not None:
    y_true = pd.to_numeric(result_df[label_col], errors="coerce")

    for _, model_row in used_models_df.iterrows():
        classifier_name = str(model_row["classifier"])
        pred_col = f"pred__{classifier_name}"
        corr_col = f"correct__{classifier_name}"

        result_df[corr_col] = np.where(
            y_true.notna(),
            (pd.to_numeric(result_df[pred_col], errors="coerce") == y_true).astype(int),
            np.nan
        )

        met = compute_binary_metrics(y_true, result_df[pred_col])
        met.update({
            "classifier": classifier_name,
            "model_id": model_row["model_id"],
            "threshold_used": model_row["threshold_used"],
            "threshold_source": model_row["threshold_source"],
            "frozen_threshold": model_row["frozen_threshold"],
        })
        metrics_rows.append(met)

metrics_df = pd.DataFrame(metrics_rows)


# ============================================================
# 7) SAVE OUTPUTS
# ============================================================

if SAVE_COMPACT_OUTPUT:
    keep_cols = [id_col]
    if label_col is not None:
        keep_cols.append(label_col)
    keep_cols += score_cols + pred_cols
    if "mean_score_across_models" in result_df.columns:
        keep_cols.append("mean_score_across_models")
    keep_cols.append("true_label")
    if label_col is not None:
        correct_cols = [c for c in result_df.columns if c.startswith("correct__")]
        keep_cols += correct_cols
    compact_df = result_df[keep_cols].copy()
else:
    compact_df = result_df.copy()

csv_out = os.path.join(OUT_DIR, "next_patients_scored.csv")
xlsx_out = os.path.join(OUT_DIR, "next_patients_scored.xlsx")
models_out = os.path.join(OUT_DIR, "used_frozen_models_summary.csv")
metrics_out = os.path.join(OUT_DIR, "threshold_based_metrics.csv")
meta_out = os.path.join(OUT_DIR, "scoring_meta.json")

compact_df.to_csv(csv_out, index=False)
compact_df.to_excel(xlsx_out, index=False)
used_models_df.to_csv(models_out, index=False)

if label_col is not None and not metrics_df.empty:
    metrics_df.to_csv(metrics_out, index=False)

meta = {
    "frozen_models_dir": FROZEN_MODELS_DIR,
    "frozen_summary_path": FROZEN_SUMMARY_PATH,
    "input_file": NEW_PATIENTS_PATH,
    "output_csv": csv_out,
    "output_xlsx": xlsx_out,
    "used_models_csv": models_out,
    "threshold_metrics_csv": metrics_out if label_col is not None and not metrics_df.empty else None,
    "n_patients_scored": int(len(compact_df)),
    "n_models_used": int(len(used_models_df)),
    "score_columns": score_cols,
    "prediction_columns": pred_cols,
    "id_column": id_col,
    "label_column": label_col,
    "threshold_override_enabled": USE_HARDCODED_THRESHOLDS,
    "hardcoded_thresholds": HARDCODED_THRESHOLDS,
}
with open(meta_out, "w") as f:
    json.dump(meta, f, indent=2)


# ============================================================
# 8) FINAL NOTEBOOK OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("SCORING COMPLETE")
print("=" * 100)
print(f"Patients scored: {len(compact_df)}")
print(f"Frozen models used: {len(used_models_df)}")
print(f"Saved CSV : {csv_out}")
print(f"Saved XLSX: {xlsx_out}")
print(f"Saved model summary: {models_out}")
if "metrics_out" in locals() and label_col is not None and not metrics_df.empty:
    print(f"Saved threshold-based metrics: {metrics_out}")
print(f"Saved meta: {meta_out}")

print("\nUsed frozen models:")
display(used_models_df)

# ------------------------------------------------------------
# SIMPLE PER-PATIENT SUMMARY TABLES (grouped by classifier)
# patient code | risk score | threshold | prediction
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("PER-PATIENT PREDICTION SUMMARY")
print("=" * 100)

# supports both old version ("threshold") and newer version ("threshold_used")
threshold_col_in_models = None
if "threshold_used" in used_models_df.columns:
    threshold_col_in_models = "threshold_used"
elif "threshold" in used_models_df.columns:
    threshold_col_in_models = "threshold"

for _, model_row in used_models_df.iterrows():
    classifier_name = str(model_row["classifier"])
    score_col = f"score__{classifier_name}"
    pred_col = f"pred__{classifier_name}"

    if score_col not in result_df.columns or pred_col not in result_df.columns:
        continue

    thr_value = model_row[threshold_col_in_models] if threshold_col_in_models is not None else np.nan

    small_view = pd.DataFrame({
        "patient_code": result_df[id_col].astype(str),
        "risk_score": pd.to_numeric(result_df[score_col], errors="coerce"),
        "threshold": float(thr_value) if pd.notna(thr_value) else np.nan,
        "prediction": pd.to_numeric(result_df[pred_col], errors="coerce"),
        "true_label": result_df["true_label"].astype(str),
    })

    # optional nice rounding for display only
    small_view["risk_score"] = small_view["risk_score"].round(6)
    small_view["threshold"] = small_view["threshold"].round(6)

    print(f"\nClassifier: {classifier_name}")
    display(small_view)

if label_col is not None and not metrics_df.empty:
    print("\nThreshold-based metrics using active thresholds:")
    display(metrics_df)

print("\nFinal scored table:")
final_display_df = compact_df.copy()

if "true_label" not in final_display_df.columns:
    final_display_df["true_label"] = result_df["true_label"].astype(str)
else:
    final_display_df["true_label"] = make_true_label_display_series(final_display_df, "true_label")

display(final_display_df.head(20))

Mounted at /content/drive
Loading frozen summary...


,classifier,model_id,save_path,threshold,val_AUC_mean,val_AUC_std,val_MCC,best_params,global_params
0,NAIVE_BAYES,no_pca__NAIVE_BAYES__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,0.071054,0.661268,0.049972,0.251787,{},"{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."
1,GAUSSIAN_PROCESS,no_pca__GAUSSIAN_PROCESS__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,0.525475,0.529863,0.099455,-0.101222,{},"{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."
2,EXTRATREES,no_pca__EXTRATREES__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,0.416637,0.649583,0.073539,0.148902,"{""clf__max_depth"": 5, ""clf__max_features"": ""sq...","{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."
3,LOGREG,no_pca__LOGREG__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,0.465771,0.671643,0.069564,0.198564,"{""clf__C"": 0.01, ""clf__max_iter"": 100, ""clf__t...","{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."
4,SGD_LOGLOSS,no_pca__SGD_LOGLOSS__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,1.000000,0.471107,0.069122,-0.099046,"{""clf__alpha"": 1e-05, ""clf__class_weight"": nul...","{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."


Loading new patient file...
Input shape: (5, 140)


,Unnamed: 0,label,Age when first GK session,Gender-female1-male2,Volumetric classification,"Hypofractionation [1=Yes, 0=No]",number of fractions,margin Dose/Fx [Gy/fx],max Dose / Fx [Gy],physical dose to margin = dose/fx x fx,...,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,1150,0,62,1,1,0,1,14,28,14,...,0.028951,0.002633,0.278346,149.063915,0.001508,0.150916,0.004944,1662.513925,0.092474,3.048629
1,1163,0,36,1,2,0,1,12,24,12,...,0.015989,0.002383,0.364186,260.504497,0.001180,0.801067,0.000929,3424.705129,0.172525,0.750654
2,1173,1,56,1,1,0,1,16,32,16,...,0.275089,0.016147,0.280639,72.177026,0.004123,0.378815,0.005886,1828.740875,0.123056,6.090834
3,1180,0,48,1,2,0,1,13,26,13,...,0.006545,0.001034,0.395837,646.094618,0.000671,0.221444,0.001207,7499.977011,0.301541,2.371035
4,1192,0,47,1,3,0,1,12,24,12,...,0.017136,0.002446,0.422716,562.921009,0.001209,0.495420,0.000538,12429.836690,0.223958,1.257061


Using ID column: Unnamed: 0
Using label column: label

[1/5] Loading frozen model: NAIVE_BAYES


...

,patient_code,risk_score,threshold,prediction,true_label
0,1150,0.0,0.8,0,0
1,1163,1.0,0.8,1,0
2,1173,1.0,0.8,1,1
3,1180,0.0,0.8,0,0
4,1192,0.0,0.8,0,0



Threshold-based metrics using active thresholds:


,n_valid,accuracy,balanced_accuracy,MCC,sensitivity,specificity,youden,TP,TN,FP,FN,classifier,model_id,threshold_used,threshold_source,frozen_threshold
0,5,1.0,1.000,1.000000,1.0,1.00,1.00,1,4,0,0,NAIVE_BAYES,no_pca__NAIVE_BAYES__G01__S01,0.98700,hardcoded:NAIVE_BAYES,0.071054
1,5,1.0,1.000,1.000000,1.0,1.00,1.00,1,4,0,0,GAUSSIAN_PROCESS,no_pca__GAUSSIAN_PROCESS__G01__S01,0.50000,hardcoded:GAUSSIAN_PROCESS,0.525475
2,5,1.0,1.000,1.000000,1.0,1.00,1.00,1,4,0,0,EXTRATREES,no_pca__EXTRATREES__G01__S01,0.56605,hardcoded:EXTRATREES,0.416637
3,5,1.0,1.000,1.000000,1.0,1.00,1.00,1,4,0,0,LOGREG,no_pca__LOGREG__G01__S01,0.70000,hardcoded:LOGREG,0.465771
4,5,0.8,0.875,0.612372,1.0,0.75,0.75,1,3,1,0,SGD_LOGLOSS,no_pca__SGD_LOGLOSS__G01__S01,0.80000,hardcoded:SGD_LOGLOSS,1.000000



Final scored table:


,Unnamed: 0,label,Age when first GK session,Gender-female1-male2,Volumetric classification,"Hypofractionation [1=Yes, 0=No]",number of fractions,margin Dose/Fx [Gy/fx],max Dose / Fx [Gy],physical dose to margin = dose/fx x fx,...,score__LOGREG,pred__LOGREG,score__SGD_LOGLOSS,pred__SGD_LOGLOSS,mean_score_across_models,correct__NAIVE_BAYES,correct__GAUSSIAN_PROCESS,correct__EXTRATREES,correct__LOGREG,correct__SGD_LOGLOSS
0,1150,0,62,1,1,0,1,14,28,14,...,0.401242,0,1.286002e-14,0,0.245259,1.0,1.0,1.0,1.0,1.0
1,1163,0,36,1,2,0,1,12,24,12,...,0.535918,0,1.000000e+00,1,0.531783,1.0,1.0,1.0,1.0,0.0
2,1173,1,56,1,1,0,1,16,32,16,...,0.896745,1,1.000000e+00,1,0.828422,1.0,1.0,1.0,1.0,1.0
3,1180,0,48,1,2,0,1,13,26,13,...,0.414290,0,6.957067e-42,0,0.262453,1.0,1.0,1.0,1.0,1.0
4,1192,0,47,1,3,0,1,12,24,12,...,0.416892,0,9.190958e-25,0,0.261269,1.0,1.0,1.0,1.0,1.0
